**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/mah-2023"], check=True)
    os.chdir("/content/encore/entries/mah-2023")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Mah et al. (2023) from DANDI:001169: models

Paper: Mah, Schiereck, Bossio and Constantinople (2023), "Distinct value computations support rapid sequential decisions", Nature Communications 14, 7573, doi:10.1038/s41467-023-43250-x.

Dataset: DANDI:001169, version 0.261120.1450 (doi:10.48324/dandi.001169/0.261120.1450), the version this repeat attempt's prompt asked for and the most recent published one. Claims file: `claims/mah-2023.yaml` (version of record with its supplement, and the paper's code at b7b68ca8c107). The data check, and the claims on behaviour alone, are in `reproduce_mah_2023.ipynb`; this notebook holds the claims that need the wait time models or simulations of the initiation time models, which take most of the run time.

| Claim | Tested here |
|---|---|
| fig3j-inferential-beats-retrospective-for-wait-times | yes |
| suppfig8-belief-state-not-distinguished | yes |
| fig3f-inferential-model-not-sensitive-to-previous-offer | yes |
| suppfig9-wait-times-reflect-misinferences | yes |
| fig3i-retrospective-model-captures-initiation | yes |
| fig3l-lambda-groups-wait-time-dynamics | yes |

## Data check

The same loading as the first notebook; its data check is repeated here in short.

In [ ]:
import os
for v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[v] = "1"  # many small fits in parallel processes: one linear-algebra thread each
import zlib, time, resource
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import mah2023 as M

def rng_for(claim, k=0, offset=0):
    """The claim's generator (protocol: seed = crc32 of '<paper key>/<claim id>'), plus `offset`
    for the seed-sensitivity runs (+1000 ... +4000) and k for a second generator."""
    return np.random.default_rng(zlib.crc32(f"mah-2023/{claim}".encode()) + k + offset)

T0 = time.time()
R = {}  # numbers quoted in the summary

In [ ]:
info = M.version_info()
assets = M.list_assets()
sessions, trials = M.load_all(assets)
print(f"loaded {len(sessions)} sessions, {len(trials)} trials in {time.time()-T0:.0f} s")

In [ ]:
trials = M.add_measures(trials)
t = trials
rats = np.sort(t["subject"].unique())
sex = sessions.drop_duplicates("subject").set_index("subject")["sex"]
print("peak memory (GB):", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6)

In [ ]:
print("Dandiset", M.DANDISET, "version", M.VERSION, "| assets:", len(assets))
print("sessions:", len(sessions), "| rats:", sessions.subject.nunique(), "| trials:", len(t))
print("opt-out catch trials with a wait time:", int(t.wt.notna().sum()), "| range (s):", t.wt.min(), "-", t.wt.max())
print("trials with a wait time (model trials):", int(t.wait_time.notna().sum()))
print("catch probability values:", sorted(t.catch_percentage.unique()), "| offered volumes:", sorted(t.reward_volume_ul.unique()))

In [ ]:
# Measures shared with the first notebook (see its sections fig2a and fig2j for their sources)
cw = t.wt.notna() & ~t.is_violation
zw = M.zscore_within(t, "wt", cw)
t["zv_wt"] = M.zscore_within(t.assign(zw=zw), "zw", cw & zw.notna(), keys=("subject", "vol"))
t["iti_dt_sess"] = M.detrend_iti(t, t.z_iti, per_session=True)
trans = M.transitions(t)
HM, LM, MH, ML = (2, 1), (3, 1), (1, 2), (1, 3)
print("transitions by type (prev -> next):", trans.groupby(["prev", "next"]).size().to_dict())

## Model fits (fig3j, suppfig8, fig3f, suppfig9)

The paper's fitting code is not public. The models are written from the Methods (Equations 1 and 3 to 10) and the generative functions in the paper's repository (BehavioralModel/WaitTimeModels at b7b68ca8c107): kappa_block = k_block C log2(min offer) / tau, D = 10 p (inferential and belief state), D = 15 p and kappa normalised to its maximum (retrospective, GenerateSynthData_AlphaPHModel.m), tau = 2.5 s, C = 1 - lowest catch probability (GenerateSynthData_Bayes.m, line 32: `C = max(1-ratTrial.prob_catch)`), offers as log2 of the deposited volume, the model run over the trials with a wait time (hits and opt-outs; line 54: `isgood = ~isnan(ratTrial.wait_time)`, the thresholded wait time of the A struct) and reset at each session. The likelihood is that of a log-normal with mean the model's wait time and variance 8 (Methods; generate_wts.m), over the opt-out catch trials. Choices this attempt makes, where the paper is silent (assumptions, see the record): bounds k in [0.01, min(0.999/C, 5)], D-scale in [0.01, 10], alpha0 in [0.001, 1]; L-BFGS-B from 2 random starts (the paper used fmincon from 100); 5 folds over sessions drawn at random; the parameters of the fold with the lowest test nLL kept (Zenodo README: final_params from the best fold), BIC and AIC from the nLL on all data (Methods). Cross-validated nLL is the summed test nLL over folds (the README's best-fold test nLL is shown as a variant). Seeds: the claim's (fig3j) and +1000 to +4000.

In [ ]:
MODEL_RATS = list(rats)
TM = time.time()
FITS = {}
for off in [0, 1000, 2000, 3000, 4000]:
    FITS[off] = M.fit_all(t, MODEL_RATS, ["inferential", "belief", "retrospective"],
                          zlib.crc32(b"mah-2023/fig3j-inferential-beats-retrospective-for-wait-times") + off, nstarts=2)
    print(f"seed +{off}: fitted {len(FITS[off])} rats, {time.time()-TM:.0f} s so far")

def metric_table(F):
    rows = []
    for r, d in F.items():
        for m, v in d.items():
            rows.append(dict(rat=r, model=m, nll=v["nll_all"], k=v["k"], n=v["n"], cv=v["cv_nll"], cvbest=v["best_fold_test_nll"]))
    df = pd.DataFrame(rows)
    df["BIC"] = np.log(df.n) * df.k + 2 * df.nll
    df["AIC"] = 2 * df.k + 2 * df.nll
    return df.pivot(index="rat", columns="model")

MT = {off: metric_table(F) for off, F in FITS.items()}
for off, mt in MT.items():
    print(f"seed +{off}, 2 starts: summed nLL over rats on all data, inferential {mt['nll']['inferential'].sum():.1f}, "
          f"belief {mt['nll']['belief'].sum():.1f}; summed test nLL, inferential {mt['cv']['inferential'].sum():.1f}, belief {mt['cv']['belief'].sum():.1f}")
P = pd.DataFrame({r: FITS[0][r]["inferential"]["params"] for r in FITS[0]}, index=["k_mixed", "k_high", "k_low", "D"]).T
print("inferential parameters (claim's seed), median:", P.median().round(3).to_dict(), "| at a bound:",
      {c: int(((P[c] <= 0.0101) | (P[c] >= 4.99)).sum()) for c in P})
print("rats whose fitted k_high > k_low (higher opportunity cost in high blocks):", int((P.k_high > P.k_low).sum()), "of", len(P))

## fig3j-inferential-beats-retrospective-for-wait-times

> Wait times are better fit by the inferential model, which uses hidden state inference to choose a block-specific value of the environment, than by the retrospective model, by BIC, AIC and cross-validated negative log-likelihood.

In [ ]:
for off, mt in MT.items():
    out = []
    for met in ["BIC", "AIC", "cv", "cvbest"]:
        d = mt[met]["inferential"] - mt[met]["retrospective"]
        p = stats.wilcoxon(d).pvalue
        out.append(f"{met}: mean diff {d.mean():+.1f}, rats inferential better {int((d < 0).sum())}/{len(d)}, p = {p:.3g}")
        R.setdefault("fig3j", {})[(off, met)] = (d.mean(), int((d < 0).sum()), p)
    print(f"seed +{off}: " + " | ".join(out))

**Result: reproduced (resting on the fitting assumptions).** Under the claim's seed the inferential model is better than the retrospective model by BIC (mean difference -1039.5, 290 of 291 rats, p = 3.6e-48), AIC (-1051.6, p = 3.6e-48) and cross-validated nLL (-454.3, 284 rats, p = 5.6e-44); under the four other seeds all three give p < 1e-45. The best-fold test nLL variant agrees (p = 4.5e-18).

## suppfig8-belief-state-not-distinguished

> Model comparison does not distinguish the inferential model from the belief state model for wait times.

A claim of absence: reproduced when none of the three signed-rank tests (BIC, AIC, cross-validated nLL) reaches p < 0.05.

In [ ]:
for off, mt in MT.items():
    out = []
    for met in ["BIC", "AIC", "cv", "cvbest"]:
        d = mt[met]["inferential"] - mt[met]["belief"]
        p = stats.wilcoxon(d).pvalue
        out.append(f"{met}: mean diff {d.mean():+.2f}, rats inferential better {int((d < 0).sum())}/{len(d)}, p = {p:.3g}")
        R.setdefault("suppfig8", {})[(off, met)] = (d.mean(), int((d < 0).sum()), p)
    print(f"seed +{off}: " + " | ".join(out))

**Result: reproduced (resting on the fitting assumptions; seed sensitive).** Under the claim's seed no test reaches p < 0.05 (BIC and AIC p = 0.163, cross-validated nLL p = 0.569). Under seed +3000 BIC and AIC give p = 0.0305 (inferential model better in 125 of 291 rats, mean difference +2.81 units), below 0.05, so the outcome is not the same under all five seeds; the other three seeds give p = 0.0684, 0.693 and 0.0624, and the cross-validated nLL gives p = 0.41 to 0.997 under all five. The best-fold test nLL variant reaches p = 0.0081 under seed +2000. The fits from 2 starts differ slightly between machines: the previous run, on another runner, gave p = 0.062 under seed +3000. Fits from 6 starts under seed +3000 (`starts_mah_2023.ipynb`) give BIC and AIC p = 0.0507 and cross-validated nLL p = 0.561, so the result stays near the threshold under that seed with more starts.

## fig3f-inferential-model-not-sensitive-to-previous-offer

> Wait times predicted by the inferential model fitted to each rat are, like the rats' wait times, not sensitive to the previous offer for 20 uL in mixed blocks.

Each rat's fitted inferential model (claim's seed) gives a wait time for every model trial; a log-normal draw with that mean and variance 8 (WTMdl, generate_wts.m) replaces the rat's wait time on its opt-out catch trials, and the analysis of fig2j is run on it (Figure3.m). Draws from the claim's generator; seeds +1000 to +4000 shown. Variants: the rats whose fitted inferential parameters are all inside their bounds, and the model's mean wait time without the draw.

In [ ]:
RMS = {r: M.RatModel(g) for r, g in t.groupby("subject") if r in set(MODEL_RATS)}

def model_wait_times(F, rng, rats_=None):
    out = pd.Series(np.nan, index=t.index)
    for r in (MODEL_RATS if rats_ is None else rats_):
        rm = RMS[r]
        w = rm.wt_opt("inferential", F[r]["inferential"]["params"], only_fit=False)
        s2 = np.log(M.NOISE_WT / w ** 2 + 1)
        draw = np.exp(rng.normal(np.log(w) - s2 / 2, np.sqrt(s2)))
        out.loc[rm.index[rm.fit]] = draw[rm.fit]
    return out

def fig2j_test(wcol):
    sel = (t.vol == 20) & (t.block == 1) & wcol.notna() & t.prev_vol.notna()
    w = t.loc[sel, ["subject", "prev_vol"]].assign(v=wcol[sel])
    w["zz"] = w.groupby("subject").v.transform(lambda x: (x - x.mean()) / x.std())
    m = pd.DataFrame({"below": w[w.prev_vol < 20].groupby("subject").zz.mean(), "above": w[w.prev_vol > 20].groupby("subject").zz.mean()})
    return M.ranksum_p(m.below, m.above), (m.below - m.above).mean(), stats.wilcoxon(m.below, m.above).pvalue

for off in [0, 1000, 2000, 3000, 4000]:
    mw_ = model_wait_times(FITS[0], rng_for("fig3f-inferential-model-not-sensitive-to-previous-offer", offset=off))
    p, sens, psr = fig2j_test(mw_)
    print(f"draw seed +{off}: model sensitivity (below minus above) {sens:+.4f}; rank-sum p = {p:.3g}; signed-rank p = {psr:.3g}")
    R.setdefault("fig3f", {})[off] = (p, sens)
p, sens, psr = fig2j_test(t.wt)
print(f"rats (fig2j analysis on the rats' wait times): sensitivity {sens:+.4f}, rank-sum p = {p:.3g}")
# variant: rats whose inferential fit has no parameter at a bound (problem 4 of the first verification)
bnd = {r: np.array(M.bounds_for("inferential", RMS[r].C)) for r in MODEL_RATS}
inner = [r for r in MODEL_RATS if not np.any(np.isclose(FITS[0][r]["inferential"]["params"], bnd[r][:, 0], atol=1e-4) |
                                                 np.isclose(FITS[0][r]["inferential"]["params"], bnd[r][:, 1], atol=1e-4))]
mw_ = model_wait_times(FITS[0], rng_for("fig3f-inferential-model-not-sensitive-to-previous-offer"), rats_=inner)
p, sens, psr = fig2j_test(mw_)
print(f"variant, {len(inner)} rats with no inferential parameter at a bound (claim's seed): model sensitivity {sens:+.4f}; rank-sum p = {p:.3g}")
# the sensitivity in the noise-free model: the model's mean wait time (Equation 1), without the log-normal draw
wmean = pd.Series(np.nan, index=t.index)
for r in MODEL_RATS:
    rm = RMS[r]
    wmean.loc[rm.index[rm.fit]] = rm.wt_opt("inferential", FITS[0][r]["inferential"]["params"], only_fit=True)
p, sens, psr = fig2j_test(wmean)
print(f"variant, the model's mean wait time without noise: sensitivity {sens:+.4f}; rank-sum p = {p:.3g}")

**Result: not reproduced (resting on the fitting assumptions; seed sensitive).** Under the claim's seed the model's 20 uL wait times are longer after offers below 20 uL (sensitivity +0.048 z, rank-sum p = 3.7e-05), while the rats' are not (-0.009, p = 0.575). Under the four other draws p is 0.332, 4.3e-07, 1.5e-05 and 4.0e-04. Leaving out the 14 rats with an inferential parameter at a bound gives +0.037, p = 1.5e-03, and the model's mean wait time without noise is more sensitive still (+0.424, p = 1.1e-61): the fitted model itself depends on the previous offer, and the draw's noise only dilutes it. Fits from 6 starts (`starts_mah_2023.ipynb`) give the same: +0.048, p = 3.6e-05 under the claim's draw, 0.328, 4.7e-07, 1.6e-05 and 4.2e-04 under the others, and +0.424, p = 9.2e-62 without noise.

## suppfig9-wait-times-reflect-misinferences

> Within mixed blocks, wait times for 20 uL differ between trials the inferential model labels as high and as low blocks, so wait times reflect mistaken inferences.

The inferential model's block labels (most likely block, Equations 3 to 6) depend on no fitted parameter (H0 is fixed), so the labels of held-out test trials are those of the full run. Per rat, mean wait time on 20 uL opt-out catch trials in mixed blocks labelled high and labelled low; signed-rank across rats.

In [ ]:
lab = pd.Series(np.nan, index=t.index)
for r in rats:
    rm = M.RatModel(t[t.subject == r])
    lab.loc[rm.index] = rm.blk_inf
sel = (t.block == 1) & (t.vol == 20) & t.wt.notna()
d = t.loc[sel, ["subject", "wt"]].assign(lab=lab[sel])
mm = d.groupby(["subject", "lab"]).wt.mean().unstack().rename(columns={0: "mixed", 1: "high", 2: "low"})
both = mm[["high", "low"]].dropna()
p = stats.wilcoxon(both.high, both.low).pvalue
print("mixed-block 20 uL catch trials by label:", d.lab.value_counts().rename({0: "mixed", 1: "high", 2: "low"}).to_dict())
print(f"rats with both labels: {len(both)}; mean wait labelled high {both.high.mean():.3f} s, labelled low {both.low.mean():.3f} s; "
      f"rats shorter when labelled high {int((both.high < both.low).sum())}; signed-rank p = {p:.3g}")
R["suppfig9"] = (both.high.mean(), both.low.mean(), p, len(both))

**Result: reproduced.** On mixed block 20 uL catch trials, rats wait 10.85 s on trials the model labels high and 11.90 s on trials it labels low (230 of 291 rats shorter on high; signed-rank p = 9.7e-23).

## fig3i-retrospective-model-captures-initiation

> A retrospective model with a learning rate scaled by changes in the inferred block posterior captures the slow and fast dynamics of trial initiation times at block transitions and their dependence on previous rewards, while a static learning rate or an unsigned prediction error gain does not.

Initiation times simulated on each rat's offers (simulate_iti in the helper): the Equation 10 model at alpha0 = 0.1042 and D-scale 0.2527 (Figure3.m), smoothing 15; the alternatives (G = 1 and G = |delta|) and the Equation 10 model at alpha0 = 0.05, 0.4 and 0.8 with smoothing 20 (Supplementary Fig. 7; FigureS7.m), D-scale 0.2527 throughout. Log-normal noise of variance 4 (Methods), 50 simulations; variance 8 (Figure3.m) as a variant. The simulated times go through the rats' pipeline (z-scored, detrended per session, transition curves). Overshoot: the high-to-mixed curve's maximum over trials 1 to 20 exceeds its value at trial 20. Persistent gap: the high-to-mixed curve lies above the low-to-mixed curve over trials 10 to 20 (mean difference > 0; a reading of "persistent"). The mean of the 50 simulations is drawn from its normal approximation; to judge small overshoots, the cell also gives each configuration without noise, under four more seeds, and with the exact mean of 50 log-normal draws.

In [ ]:
SIMC = ["subject", "sess", "trial", "block", "vol", "reward_volume_ul", "wait_time", "inter_trial_interval"]
by_rat = {r: g for r, g in t[SIMC].groupby("subject")}

def sim_curves(model, alpha0, noise, smooth, rng, dscale=0.2527, exact=False):
    sim = pd.concat([M.simulate_iti(by_rat[r], model, alpha0, dscale, noise, rng, exact=exact) for r in MODEL_RATS])
    tt = t.loc[sim.index, ["subject", "sess", "trial", "block"]].assign(sim=sim)
    v = tt.sim.where(tt.trial > 0)
    z = (v - v.groupby(tt.subject).transform("mean")) / v.groupby(tt.subject).transform("std")
    tt["dt"] = M.detrend_iti(tt, z, per_session=True)
    tt = tt.reset_index(drop=True)
    c = M.transition_curves(tt, tt.dt, M.transitions(tt), smooth=smooth)
    h, l = c[HM].mean(), c[LM].mean()
    over = float(h.loc[1:20].max() - h.loc[20])
    gap_ = float((h - l).loc[10:20].mean())
    return over, gap_, h, l

FIG3I = {}
configs = [("eq10", 0.1042, 15)] + [(m, a, 20) for m in ["static", "rpe", "eq10"] for a in [0.05, 0.4, 0.8]]
for noise in [4, 8]:
    rng = rng_for("fig3i-retrospective-model-captures-initiation", offset=0 if noise == 4 else 1)
    for model, a0, sm in configs:
        over, gap_, h, l = sim_curves(model, a0, noise, sm, rng)
        FIG3I[(noise, model, a0)] = (over, gap_)
        print(f"noise {noise}: {model:6s} alpha0 {a0:<6}: overshoot {over:+.4f} ({'yes' if over > 0 else 'no'}), "
              f"gap after high minus after low, trials 10-20 {gap_:+.4f} ({'yes' if gap_ > 0 else 'no'})")
R["fig3i"] = FIG3I
# Noise floor of the overshoot. (1) The models without noise: the overshoot of the curve itself.
# (2) Noise 4 under the claim's seed + 1000 ... + 4000. (3) The exact mean of 50 log-normal draws
# per trial in place of its normal approximation, noise 4, claim's seed.
FLOOR = {}
for model, a0, sm in configs:
    over0, gap0, _, _ = sim_curves(model, a0, 0.0, sm, rng_for("fig3i-retrospective-model-captures-initiation", k=2))
    seeds = [sim_curves(model, a0, 4, sm, rng_for("fig3i-retrospective-model-captures-initiation", offset=o))[0] for o in [1000, 2000, 3000, 4000]]
    FLOOR[(model, a0)] = (over0, gap0, seeds)
    print(f"{model:6s} alpha0 {a0:<6}: without noise overshoot {over0:+.5f}, gap {gap0:+.4f}; noise 4, seeds +1000..+4000 overshoot "
          + ", ".join(f"{s:+.4f}" for s in seeds) + f" (claim's seed {FIG3I[(4, model, a0)][0]:+.4f})")
for model, a0, sm in [c for c in configs if c[1] in (0.1042, 0.4, 0.8) and c[0] != "eq10" or c[1] == 0.1042]:
    over, gap_, _, _ = sim_curves(model, a0, 4, sm, rng_for("fig3i-retrospective-model-captures-initiation", k=3), exact=True)
    print(f"exact mean of 50 log-normal draws, noise 4: {model:6s} alpha0 {a0:<6}: overshoot {over:+.4f}, gap {gap_:+.4f}")

In [ ]:
# Fig. 3i: each rat's retrospective parameters fitted to wait times (fig3j, claim's seed), simulated
# initiation times (one draw, noise 1.5 as Figure3.m, and 4 as the Methods), then the analysis of
# fig2k (code reading: all mixed block trials; and the Methods reading: 20 uL trials)
for noise in [4, 1.5]:
    rng = rng_for("fig3i-retrospective-model-captures-initiation", k=1, offset=0 if noise == 4 else 1)
    sim = pd.concat([M.simulate_iti(by_rat[r], "eq10", FITS[0][r]["retrospective"]["params"][0],
                                    FITS[0][r]["retrospective"]["params"][1], noise, rng, nsims=1) for r in MODEL_RATS])
    tt = t.loc[sim.index, ["subject", "trial", "block", "vol", "prev_vol"]].assign(sim=sim.where(t.loc[sim.index, "trial"] > 0))
    tt["z"] = (tt.sim - tt.groupby("subject").sim.transform("mean")) / tt.groupby("subject").sim.transform("std")
    for name, m in [("C: all mixed trials", tt.block == 1), ("M: 20 uL trials", (tt.block == 1) & (tt.vol == 20))]:
        d = tt[m & tt.prev_vol.notna() & tt.z.notna()]
        mk = pd.DataFrame({"below": d[d.prev_vol < 20].groupby("subject").z.mean(), "above": d[d.prev_vol > 20].groupby("subject").z.mean()}).dropna()
        p = stats.wilcoxon(mk.below, mk.above).pvalue
        print(f"noise {noise}, {name}: model mean z after < 20 {mk.below.mean():+.4f}, after > 20 {mk.above.mean():+.4f}; "
              f"rats higher after smaller offers {int((mk.below > mk.above).sum())} of {len(mk)}; signed-rank p = {p:.3g}")
        R.setdefault("fig3i_prev", {})[(noise, name)] = (mk.below.mean(), mk.above.mean(), p)

**Result: partially reproduced.** The Equation 10 model at alpha0 = 0.1042 shows both an overshoot (0.200 z) and a persistent gap (+1.398 z over trials 10-20), and its initiation times, simulated with each rat's fitted parameters, are slower after offers below 20 uL (+0.015 against -0.009 z, 232 of 291 rats, p = 1.8e-29). The static and |delta| models lack both at alpha0 = 0.05, but at 0.4 and 0.8 each shows a small overshoot (0.0002 to 0.0058 z) and a gap (+0.09 to +0.13), so by the criterion's rule they are not excluded at every alpha0. The overshoot is not noise: without noise these models give 0.0005 to 0.0073 z (the Equation 10 model 0.280), four more seeds give 0 to 0.0059, and the exact mean of 50 draws gives 0.0008 to 0.0066. It is at most 1/38 of the Equation 10 model's overshoot without noise; the criterion gives no size below which an overshoot would count as absent. Noise of variance 8 gives the same pattern.

## fig3l-lambda-groups-wait-time-dynamics

> Rats with optimal and poor inference (high or low lambda of the sub-optimal inferential model) differ in their wait time dynamics at transitions from mixed into low or high blocks, but not in their trial initiation time dynamics.

Lambda per rat from the sub-optimal inferential model (Equation 8; GenerateSynthData_Bayes_subopt_onlylambda_flat.m: five parameters, belief reset to flat at each session). For time, the model is fitted once on all of each rat's data from one random start (claim's seed), without the cross-validation of fig3j (a deviation). Groups below the 20th and above the 80th percentile of lambda (Methods, Results) and below the 40th and above the 60th (Figure3.m). Mixed-to-low and mixed-to-high curves (wait time z-scored per offer, initiation time detrended per session, 30 trials either side, smoothed over 10, trials -20 to 20 kept) averaged per group; Equation 11 fitted to each group mean with the bounds of my_fit_sigmoid.m (C in [0, 5], so A is the left and D the right asymptote; x0 in [0, 30]; A and D in [-5, 5]) and continuous parameters (a grid start refined by bounded least squares, M.fit_logistic); 500 label shuffles, each half refitted; p = share of shuffles with |parameter difference| above the observed, divided by 2 (myLambdaShuffleTest.m, written as myTauShuffleTest.m). Shuffle seeds: the claim's and +1000 to +4000. The cell prints the fits, the shuffles tied with the observed difference and the shuffle fits at a bound.

In [ ]:
TL = time.time()
LAM = M.fit_all(t, MODEL_RATS, ["lambda"], zlib.crc32(b"mah-2023/fig3l-lambda-groups-wait-time-dynamics"), nstarts=1, cv=False)
lam = pd.Series({r: LAM[r]["lambda"]["params"][4] for r in MODEL_RATS})
print(f"lambda fits: {time.time()-TL:.0f} s; lambda median {lam.median():.3f}, range {lam.min():.3f}-{lam.max():.3f}; "
      f"at 0: {int((lam <= 1e-6).sum())}, at 1: {int((lam >= 1 - 1e-6).sum())}")
c30 = {k: M.transition_curves(t, t[k], trans, twin=30, rats=MODEL_RATS) for k in ["zv_wt", "iti_dt_sess"]}
xs = np.arange(-20, 21)
PAR = {"A": 0, "D": 1, "C": 2, "x0": 3}

def lam_test(curves, lo_q, hi_q, rng, nshuf=500):
    """myLambdaShuffleTest.m; the logistic fit has the bounds of my_fit_sigmoid.m (C >= 0, so A is
    the left and D the right asymptote) and continuous parameters (M.fit_logistic)."""
    sel = lo_q | hi_q
    Cv = curves.loc[sel.index[sel], -20:20].to_numpy()
    n = len(Cv); h = n // 2
    g1 = np.nanmean(curves.loc[lo_q.index[lo_q], -20:20].to_numpy(), 0)
    g2 = np.nanmean(curves.loc[hi_q.index[hi_q], -20:20].to_numpy(), 0)
    f1, f2 = M.fit_logistic(xs, g1)[0], M.fit_logistic(xs, g2)[0]
    obs = f1 - f2
    perms = np.array([rng.permutation(n) for _ in range(nshuf)])
    a = M.fit_logistic(xs, np.nanmean(Cv[perms[:, :h]], 1)); b = M.fit_logistic(xs, np.nanmean(Cv[perms[:, h:]], 1))
    sh = a - b
    p = (np.abs(sh) > np.abs(obs)).sum(0) / (2 * nshuf)
    ties = np.isclose(np.abs(sh), np.abs(obs), rtol=0, atol=1e-9).sum(0)
    ab = np.r_[a, b]
    at_bound = {"C at 0": int((ab[:, 2] <= 1e-6).sum()), "C at 5": int((ab[:, 2] >= 5 - 1e-6).sum()), "x0 at 0": int((ab[:, 3] <= 1e-6).sum()),
                "x0 at 30": int((ab[:, 3] >= 30 - 1e-6).sum())}
    return dict(zip(PAR, p)), dict(zip(PAR, obs)), (f1, f2), dict(zip(PAR, ties)), at_bound

for split in [(20, 80), (40, 60)]:
    lo_q = lam < np.percentile(lam, split[0]); hi_q = lam > np.percentile(lam, split[1])
    print(f"split {split}: low-lambda group {int(lo_q.sum())}, high-lambda group {int(hi_q.sum())} rats")
    for off in ([0, 1000, 2000, 3000, 4000] if split == (20, 80) else [0]):
        for meas in ["zv_wt", "iti_dt_sess"]:
            for tr, nm in [(ML, "mixed->low"), (MH, "mixed->high")]:
                rng = rng_for("fig3l-lambda-groups-wait-time-dynamics", offset=off)
                p, obs, fits, ties, atb = lam_test(c30[meas][tr], lo_q, hi_q, rng)
                R.setdefault("fig3l", {})[(split, off, meas, nm)] = p
                print(f"   seed +{off} {meas:12s} {nm:12s} p: " + ", ".join(f"{k} {v:.3f}" for k, v in p.items())
                      + " | observed differences: " + ", ".join(f"{k} {v:+.3f}" for k, v in obs.items()))
                if off == 0:
                    print("        fits (A, D, C, x0): low lambda", np.round(fits[0], 3).tolist(), "high lambda", np.round(fits[1], 3).tolist(),
                          "| shuffles tied with the observed |difference|:", ties, "| shuffle fits at a bound (of 1000):", atb)
print(f"fig3l total {time.time()-TL:.0f} s")

**Result: not reproduced (resting on the lambda fit, an assumption).** With the 20/80 split (58 rats per group), the wait time slope C does not differ at mixed to low (p = 0.337; 0.337 to 0.364 over the five seeds), nor C (p = 0.332; 0.319 to 0.342) or the right asymptote D (p = 0.335; 0.330 to 0.357) at mixed to high, so neither the Results' reading nor the legend's holds. The wait time right asymptote at mixed to low, which the criterion does not test, gives p = 0.021. The initiation time slope C at mixed to low gives p = 0.047 (0.032 to 0.057 over the seeds), and at mixed to high C p = 0.320 and D p = 0.314. Under the claim's seed no observed difference of a tested parameter is tied with a shuffle, and no shuffle fit has C at either bound (0 or 5). The 40/60 split of Figure3.m gives p = 0.178 for wait time C at mixed to low, 0.472 and 0.224 for C and D at mixed to high, and 0.012 for initiation time C at mixed to low. In both groups at mixed to high, the initiation time midpoint x0 sits at its lower bound 0. The lambda fits from one start differ slightly between machines (4 rats at lambda 0 here, 3 in the previous run on another runner, where wait time C at mixed to low gave p = 0.453); both runs give the same outcome.

## Summary

| Claim | Outcome |
|---|---|
| fig3j-inferential-beats-retrospective-for-wait-times | reproduced, resting on the fitting assumptions |
| suppfig8-belief-state-not-distinguished | reproduced, resting on the fitting assumptions; seed sensitive |
| fig3f-inferential-model-not-sensitive-to-previous-offer | not reproduced, resting on the fitting assumptions; seed sensitive |
| suppfig9-wait-times-reflect-misinferences | reproduced |
| fig3i-retrospective-model-captures-initiation | partially reproduced |
| fig3l-lambda-groups-wait-time-dynamics | not reproduced, resting on the lambda fit |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the paper's fitting code is not public, so the bounds, the number of starts (2 here, 100 in the paper; 6 starts for suppfig8 and fig3f in `starts_mah_2023.ipynb`), the folds (random over sessions) and the choice of the cross-validated nLL are this attempt's; the models themselves follow the Methods and the paper's generative code. The lambda model is fitted once on all data from one start, without cross-validation. Simulated initiation times use the normal approximation to the mean of 50 log-normal draws (the exact mean is shown for fig3i). The logistic of fig3l is fitted with the bounds of my_fit_sigmoid.m by a grid start and bounded least squares, in place of fmincon from 10 random starts. The paper's deposited fits on Zenodo were not used.